# Лабораторная работа № 1. Линейные методы

### Набор данных

- Выберите любой набор данных для задачи бинарной классификации. Если он был не для бинарной классификации, то объедините несколько классов.
- Преобразуйте его в числовой вид и нормализуйте.
- Разбейте его на тренировочную и тестовую часть.
- Выберите целевую функцию ошибки или качества для задачи бинарной классификации.
- Если после преобразования в числа в наборе данных слишком много признаков, выберите часть признаков из набора данных любым способом. Например, при помощи абсолютной корреляции с целевым (с заменой на ±1).

### Алгоритмы

- Реализуйте алгоритм линейной регрессии в матричном виде с гребневой регуляризацией. Можно использовать библиотеки для работы с матрицами. Преобразуйте его в алгоритм линейной классификации через замену целевого признака на ±1.
- Реализуйте алгоритм линейной классификации на основе градиентного спуска.
- Алгоритм должен поддерживать не менее трёх эмпирических рисков, Elastic Net регуляризацию и настраиваемую скорость градиентного спуска. Эмпирические риски должны быть именно для задачи линейной классификации, вычисляться через отступ. Производные и градиент должны быть вычислены аналитически.

### Задача

- Постарайтесь подобрать хорошие гиперпараметры для каждого алгоритма.
- Постройте кривую обучения со сглаженным эмпирическим риском на тренировочном множестве для линейной классификации.
- Постройте кривую обучения с целевой функцией ошибки или качества на тестовом множестве для линейной классификации. Не обязательно замерять целевую функцию на каждой итерации, если их много. Переберите разные разбиения на тренировочную и тестовую часть как при перекрёстной проверке. Отметьте доверительный интервал на графике.
- Отметьте на предыдущем графике значение целевой функции на тестовом множестве для линейной регрессии. Например, горизонтальной прямой.
- Постройте график зависимости значений коэффициентов линейной модели от значения коэффициента L1-регуляризации (коэффициент для L2 должен быть равен нулю). Постройте второй график для зависимости от коэффициента L2-регуляризации (для L1 коэффициент равен нулю). Желательно перебирать коэффициенты регуляризации в логарифмированном пространстве.

## Подготовка данных

Воспользуемся датасетом [Selfie2Anime](https://huggingface.co/datasets/huggan/selfie2anime), в нём 3500 фотографий женских лиц с меткой $-1$ и 3500 изображений женских аниме-персонажей с меткой $+1$. Исходные разбиения датасета объединил и сделал собственное разбиение 80/20 (обучающая/тестовая).

При первом запуске исходные Parquet-файлы скачиваются в `dataset/original`; при следующих запусках повторное скачивание не требуется. Каждое изображение переводится в RGB, уменьшается с 256×256 до 32×32, разворачивается в вектор из $32\cdot32\cdot3=3072$ признаков, все значения лежат в отрезке от 0 до 255, поэтому отнормируем поделив на 255.

Признаков слишком много (не охото долго ждать вычисления), поэтому давайте уменьшим их кол-во до 333 методом абсолютной корреляции с целевым (чем больше модуль, тем более явно признак разграничивает две категории объектов).

В `dataset/after_normalized` сохраняются готовые `train.npz` и `test.npz`, общая числовая выборка для будущей перекрёстной проверки.


## Целевая метрика качества

Для оценки и сравнения моделей выбрана **accuracy** — кол-во цспешных предсказаний делим на размер тестовой выборки.

Линейная модель вычисляет оценку $a(x)=w^Tx+b$, и мы считаем классом $+1$ (аниме), если $a(x)\geq0$, и классом $-1$ (реальный человек), если $a(x)<0$. Стремимся увеличить accuracy по максимуму.



In [1]:
from pathlib import Path

from utils.download import download_dataset
from utils.normalization import prepare_dataset

DATASET_DIR = Path("dataset")
REPOSITORY_ID = "huggan/selfie2anime"
FILENAMES = [
    "data/train-00000-of-00001.parquet",
    "data/test-00000-of-00001.parquet",
]
IMAGE_SIZE = 32
TEST_SIZE = 0.2
RANDOM = 24422442
FEATURES_COUNT = 333

parquet_paths = download_dataset(
    REPOSITORY_ID,
    FILENAMES,
    DATASET_DIR / "original",
)

dataset_info = prepare_dataset(
    parquet_paths,
    DATASET_DIR / "after_normalized",
    IMAGE_SIZE,
    TEST_SIZE,
    RANDOM,
    FEATURES_COUNT,
)
dataset_info


Датасет уже скачан.


{'all_samples': 7000,
 'train_shape': (5600, 333),
 'test_shape': (1400, 333),
 'classes': {-1: 'фотография', 1: 'аниме'}}